# Generiká

Dôležitý koncept, nielen v C#, ale v celom .NET-e i iných jazykov.
Umožňuje:
- zvýšenie výkonu
- typovú bezpečnosť
- viacnásobné použitie raz napísaného kódu

## Test výkonu

Jedna z najdôležitejších výhod generík je **výkon** (rýchlosť spracovania dát). Pozrime sa na použitie negenerickej verzie zoznamu typu `ArrayList` a jej generickej verzie `List<T>`.  
Prerekvizita:

In [ ]:
using System.Diagnostics;
var stopwatch = new Stopwatch();

// Počet iterácií (aby boli výsledky hodnoverné)
const int IterationCount = 20;
// Počet čísiel, ktoré sa pridajú do zoznamu a prečítajú zo zoznamu (negenerického / generického)
const int MaxNumberCount = 10_000_000;

var boxingTimes = new List<double>();
var unboxingTimes = new List<double>();

Test rýchlosti negenerickej triedy `ArrayList`:

In [ ]:
boxingTimes.Clear();
unboxingTimes.Clear();

for (int c = 0; c < IterationCount; c++)
{
    stopwatch.Restart();
    var list = new ArrayList();
    for (int i = 0; i < MaxNumberCount; i++) 
    {
        list.Add(i); // Tu dochádza k boxingu - konverzia hodnotového typu (int) na referenčný typ (object)
    }    
    boxingTimes.Add(stopwatch.ElapsedMilliseconds);
    
    stopwatch.Restart();
    var result = new StringBuilder(MaxNumberCount); 
    for (int i = 0; i < MaxNumberCount; i++) 
    {
        int number = (int)list[i]; // Tu dochádza k unboxingu - konverzia z referenčného typu (object) na hodnotový typu (int)
        result.Append(number); 
    }    
    unboxingTimes.Add(stopwatch.ElapsedMilliseconds);
}

Console.WriteLine($"ArrayList - pridanie cisiel trvalo priemerne (boxing): {boxingTimes.Average()}");
Console.WriteLine($"ArrayList - precitanie cisiel trvalo priemerne (unboxing): {unboxingTimes.Average()}");

Test rýchlosti generickej triedy `List<int>`:

In [ ]:
boxingTimes.Clear();
unboxingTimes.Clear();

for (int c = 0; c < IterationCount; c++)
{
    stopwatch.Restart();
    var list = new List<int>();
    for (int i = 0; i < MaxNumberCount; i++) 
    {
        list.Add(i); // Tu k boxingu nedochádza
    }    
    boxingTimes.Add(stopwatch.ElapsedMilliseconds);
    
    stopwatch.Restart();
    var result = new StringBuilder(MaxNumberCount); 
    for (int i = 0; i < MaxNumberCount; i++) 
    {
        int number = list[i]; // Tu k unboxingu nedochádza
        result.Append(number); 
    }    
    unboxingTimes.Add(stopwatch.ElapsedMilliseconds);
}

Console.WriteLine($"List - pridanie cisiel trvalo priemerne: {boxingTimes.Average()} ms");
Console.WriteLine($"List - precitanie cisiel trvalo priemerne: {unboxingTimes.Average()} ms");

## Typová bezpečnosť 

Do negenerického ArrayListu môže byť pridaný akýkoľvek objekt iného typu ako požadovaného. Ak to chceme, je to fajn. Ak by sme chceli prejsť všetky položky foreach-om, a skonvertovať ich na nejaký typ, môžeme dostať chybu za behu (výnimka `InvalidCastException`):


In [ ]:
var numbers = new ArrayList();
numbers.Add(1);
numbers.Add(2);
numbers.Add(3);
numbers.Add("4");
numbers.Add(3);

foreach (int number in numbers)
{
    Console.WriteLine(number);    
}

Pri generickej verzii dostaneme kompilačnú chybu (`Error CS1503`), takže ju môžeme hneď vyriešiť:

In [ ]:
var numbers = new List<int>();
numbers.Add(1);
numbers.Add(2);
numbers.Add(3);
numbers.Add("4"); // Error CS1503: Argument 1: cannot convert from 'string' to 'int'
numbers.Add(3);

## Znovupoužitie binárneho kódu

Výhoda generík je, že ich môžeme definovať raz a potom ich použiť s viacerými typmi.  
Generiká môžu byť definované v jednom jazyku a potom použité v akomkoľvek inom .NET-ovom jazyku.  
Príklad použitia triedy `List` s rôznymi typmi:


In [ ]:
var numbers = new List<int>();
numbers.Add(10);

var realNumbers = new List<double>();
realNumbers.Add(Math.PI);
realNumbers.Add(10);
realNumbers.Add(1.123);

var words = new List<string>();
words.Add("Hello");
words.Add("generics");

class Person 
{
    public string FullName {get; set;}
    public DateTime Birthdate { get; set; }  

    public Person(string fullName, DateTime birthdate) => (FullName, Birthdate) = (fullName, birthdate);  
}

var people = new List<Person>();
people.Add(new Person("Jan Mrkvicka", DateTime.Today));
people.Add(new Person("Elena Mrkvickova", DateTime.Now.AddYears(-20)));
foreach(var p in people)
    Console.WriteLine($"{p.FullName} ({p.Birthdate})");

V IL kóde je definícia generickej triedy iba raz. Až JIT kompilátor (CLR) kompiluje každú triedu s daným použitým typom do natívneho kódu, pričom tu záleží od toho, či sa jedná o referenčný alebo hodnotový typ.  
Pri hodnotových typoch JIT kompilátor vytvára novú triedu pre konkrétny použitý typ. Pri referenčných typoch je trieda iba jedna, ktorá sa zdieľa. Je to preto, že veľkosť referencie na ľubovoľný referenčný typ je vždy rovnaký – t. j. buď 32 bitov (4 bajty) alebo 64 bitov (8 bajtov) podľa toho, či beží ako 32-bitový alebo 64-bitovú proces. Avšak, pri hodnotových typov je veľkosť pamäte rôzna, preto sa vytvára každá samostatná trieda.

## Pomenovanie 

- Názov prefixu generického typu začína na písmeno T
- Ak generický typ môže byť nahradený akýmkoľvek typom a je iba jeden typ použitý, potom sa používa ako názov generického typu iba T (List<T>, LinkedList<T>)
- Ak je špeciálna požiadavka (nastavené obmedzenie) na generický typ alebo ak sú použité dva a viac typov, používa sa popisný názov začínajúci na písmenko T (TEventArgs, TInput, TKey, TValue, …)


## Generické metódy

In [ ]:
static void ProcessItems<T>(IList<T> list)
{
    System.Console.WriteLine($"IsReadOnly vracia {list.IsReadOnly} pre {list.GetType()}.");

    //coll.RemoveAt(4);

    foreach (T item in list)
    {
        System.Console.Write(item.ToString() + " ");
    }
    System.Console.WriteLine();
}

int[] arr = { 0, 1, 2, 3, 4 };
List<int> list = new List<int>();

for (int x = 5; x < 10; x++)
{
    list.Add(x);
}

ProcessItems<int>(arr);
ProcessItems<int>(list);